#### Import libraries

In [4]:
import glob
import re
import codecs
import pandas as pd


#### Read files using `glob`
- Code below will store all filenames in directory ending in `.txt.txt` into `filelist`
- for loop below is just printing elements of the list to confirm this is working as expected.


In [3]:
filelist = glob.glob('*.txt.txt')
for file in filelist:
    print(file)

1901302120_H5_D_13.txt.txt
1901302235_H6_S_14.txt.txt
1901302225_H12_D_13.txt.txt
1901302121_H8_S_14.txt.txt
1901302158_H9_S_13.txt.txt
1901302217_H18_D_14.txt.txt
1901302236_H17_D_13.txt.txt
1901302110_H13_S_14.txt.txt
1901302146_H14_D_14.txt.txt
1901302222_H2_S_13.txt.txt
1901302240_H8_D_14.txt.txt
1901302109_H7_S_14.txt.txt
1901302212_H9_D_13.txt.txt
1901302150_H14_S_14.txt.txt
1901302119_H4_S_13.txt.txt
1901302141_H19_S_14.txt.txt
1901302241_H2_D_14.txt.txt
1901302203_H7_S_13.txt.txt
1901302118_H1_D_13.txt.txt
1901302194_H10_S_14.txt.txt
1901302117_H15_S_13.txt.txt
1901302227_H11_S_14.txt.txt
1901302224_H3_S_13.txt.txt
1901302237_H10_D_13.txt.txt
1901302136_H15_D_14.txt.txt
1901302108_H7_D_13.txt.txt
1901302115_H1_S_13.txt.txt
1901302138_H11_D_13.txt.txt
1901302228_H4_D_13.txt.txt
1901302223_H16_S_14.txt.txt
1901302134_H13_D_13.txt.txt
1901302157_H12_S_13.txt.txt
1901302214_H19_D_13.txt.txt
1901302243_H5_S_14.txt.txt
1901302122_H3_D_14.txt.txt
1901302137_H18_S_13.txt.txt
1901302202

In [ ]:
import glob, os, io
import pandas as pd

def read_clean_df(path):
    with open(path, "rb") as f:
        raw = f.read().replace(b"\x00", b"")   # remove NULLs
    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        text = raw.decode("latin-1")
    return pd.read_csv(io.StringIO(text), sep=None, engine="python")

filelist = glob.glob("*.txt.txt")
print("Found", len(filelist), "files")

wide_frames = []
for fn in filelist:
    df = read_clean_df(fn)

    # Build a timestamp key if Date/Time exist (adjust to your columns)
    if {"Date", "Time"}.issubset(df.columns):
        key = pd.to_datetime(df["Date"] + " " + df["Time"], errors="coerce")
    elif "Date Time" in df.columns:
        key = pd.to_datetime(df["Date Time"], errors="coerce")
    else:
        # Fall back to row number if no obvious key (least ideal)
        key = pd.RangeIndex(len(df))

    df_keyed = df.copy()
    df_keyed.index = key

    # Keep a single value column per file (adjust if you want a different one)
    value_col = df_keyed.columns[-1]  # e.g., last column is the reading
    col_name = os.path.splitext(os.path.splitext(os.path.basename(fn))[0])[0]
    df_single = df_keyed[[value_col]].rename(columns={value_col: col_name})

    wide_frames.append(df_single)

# Outer-join on the key (keeps all timestamps that appear in any file)
merged = pd.concat(wide_frames, axis=1)

# Optional: sort by time if the index is a datetime
if pd.api.types.is_datetime64_any_dtype(merged.index):
    merged = merged.sort_index()

merged.to_csv("cbind_by_key.csv", index=True)  # keep index (timestamp) in the file
print("Wrote cbind_by_key.csv with shape:", merged.shape)
